# $\pi^0$ Dalitz decay: Geant4 vs EvtGen

Compares the $\pi^0 \to e^+ e^- \gamma$ Dalitz decay kinematics between:

- **Geant4** truth decays from the BNB `nu_overlay` file (extracted in `pi0_dalitz_decays.ipynb` -> `pi0_dalitz_momenta.npz`), boosted into the $\pi^0$ rest frame, and
- **EvtGen** $\pi^0$ Dalitz decays generated *at rest* with the `PI0_DALITZ` model (Kroll-Wada QED matrix element with form factor), from `evtgen/pi0_dalitz_gen.cpp`.

All observables are computed in the $\pi^0$ rest frame. The most sensitive observable to the decay matrix element is the $e^+e^-$ invariant mass $m_{ee}$ (driven by the $1/m_{ee}^2$ virtual-photon propagator); the Geant4 treatment is suspected of being less accurate than the full QED expression.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

M_PI0 = 134.9768  # MeV
M_E = 0.51099895

GEANT4_NPZ = "pi0_dalitz_momenta.npz"
EVTGEN_CSV = "/nevis/riverside/data/leehagaman/ngem/data_files/evtgen_pi0_dalitz.csv"

PLOT_DIR = "../plots/pi0_dalitz"
os.makedirs(PLOT_DIR, exist_ok=True)


def savefig(name):
    plt.savefig(os.path.join(PLOT_DIR, name + ".pdf"))
    plt.savefig(os.path.join(PLOT_DIR, name + ".png"), dpi=200)

In [ ]:
# four-vectors as [E, px, py, pz]
def boost_to_rest(p4, frame_p4):
    p4 = np.asarray(p4, dtype=float)
    frame_p4 = np.asarray(frame_p4, dtype=float)
    E_frame = frame_p4[..., 0]
    beta = frame_p4[..., 1:] / E_frame[..., None]
    b2 = np.sum(beta**2, axis=-1)
    gamma = 1.0 / np.sqrt(1.0 - b2)
    pe = p4[..., 0]
    pv = p4[..., 1:]
    bp = np.sum(beta * pv, axis=-1)
    with np.errstate(divide="ignore", invalid="ignore"):
        gamma2 = np.where(b2 > 0, (gamma - 1.0) / b2, 0.0)  # ->0 for a frame already at rest
    E_prime = gamma * (pe - bp)
    pv_prime = pv + (gamma2 * bp - gamma * pe)[..., None] * beta
    return np.concatenate([E_prime[..., None], pv_prime], axis=-1)


def p3(p4):
    return p4[..., 1:]


def minkowski_mass(p4):
    return np.sqrt(np.clip(p4[..., 0]**2 - np.sum(p4[..., 1:]**2, axis=-1), 0, None))


def angle_between(a3, b3):
    cos = np.sum(a3 * b3, axis=-1) / (np.linalg.norm(a3, axis=-1) * np.linalg.norm(b3, axis=-1))
    return np.clip(cos, -1.0, 1.0)


def observables(gamma, ep, em):
    """Given lab (or any-frame) four-vectors, return Dalitz observables in the pi0 rest frame."""
    pi0 = gamma + ep + em
    g = boost_to_rest(gamma, pi0)
    p = boost_to_rest(ep, pi0)
    m = boost_to_rest(em, pi0)
    out = {}
    out["E_gamma"] = g[:, 0]
    out["E_ep"] = p[:, 0]
    out["E_em"] = m[:, 0]
    out["m_ee"] = minkowski_mass(p + m)                       # frame-independent
    out["opening"] = np.degrees(np.arccos(angle_between(p3(p), p3(m))))
    out["asym"] = (p[:, 0] - m[:, 0]) / (p[:, 0] + m[:, 0])
    # Dalitz lepton angle (helicity frame): e+ direction measured in the e+e- rest
    # frame, relative to the dilepton's flight direction in the pi0 rest frame
    # (= the boost axis between the two frames). The dilepton is at rest in its own
    # frame, so the reference axis must be taken from the pi0 frame, not the ee frame.
    ee = p + m
    ep_in_ee = boost_to_rest(p, ee)
    out["cos_theta_l"] = angle_between(p3(ep_in_ee), p3(ee))
    return out

In [ ]:
# ---- Geant4 sample ----
g4 = np.load(GEANT4_NPZ)
g4_obs = observables(g4["gamma_p4"], g4["ep_p4"], g4["em_p4"])
print(f"Geant4 Dalitz decays: {len(g4_obs['m_ee']):,}")

# ---- EvtGen sample ----
ev = pd.read_csv(EVTGEN_CSV)
ev_gamma = ev[["E_g", "px_g", "py_g", "pz_g"]].values
ev_ep = ev[["E_ep", "px_ep", "py_ep", "pz_ep"]].values
ev_em = ev[["E_em", "px_em", "py_em", "pz_em"]].values
ev_obs = observables(ev_gamma, ev_ep, ev_em)
print(f"EvtGen Dalitz decays: {len(ev_obs['m_ee']):,}")

## Overlay comparison

Histograms are area-normalized (`density=True`) so the shapes can be compared directly despite the very different statistics. All observables are in the $\pi^0$ rest frame (denoted by a star, e.g. $E^{*}$).

- **$\cos\theta_\ell$** is the Dalitz lepton angle: the $e^+$ direction measured in the $e^+e^-$ rest frame, relative to the dilepton's line of flight in the $\pi^0$ rest frame. The QED matrix element predicts a $(1+\cos^2\theta_\ell)$ shape.
- The **$m_{ee}$** panel (bottom-right) is log-log; the next section enlarges that same distribution.

In [ ]:
def compare(key, bins, xlabel, logy=False, logx=False, ax=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(6, 4))
    ax.hist(g4_obs[key], bins=bins, density=True, histtype="step", lw=1.8,
            color="tab:blue", label="Geant4 (BNB overlay)")
    ax.hist(ev_obs[key], bins=bins, density=True, histtype="step", lw=1.8,
            color="tab:red", label="EvtGen PI0_DALITZ")
    ax.set_xlabel(xlabel)
    ax.set_ylabel("normalized")
    if logy:
        ax.set_yscale("log")
    if logx:
        ax.set_xscale("log")
    ax.legend(fontsize=8)
    return ax


mee_bins = np.logspace(np.log10(2 * M_E), np.log10(M_PI0), 60)

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
compare("E_gamma", np.linspace(0, 68, 60), r"$E^{*}_{\gamma}$ [MeV] ($\pi^0$ rest frame)", ax=axes[0, 0])
compare("E_ep", np.linspace(0, 68, 60), r"$E^{*}_{e^+}$ [MeV] ($\pi^0$ rest frame)", ax=axes[0, 1])
compare("opening", np.linspace(0, 180, 60), r"$e^+e^-$ opening angle [deg] ($\pi^0$ rest frame)", ax=axes[0, 2])
compare("asym", np.linspace(-1, 1, 60), r"energy asymmetry $(E^{*}_{e^+}-E^{*}_{e^-})/(E^{*}_{e^+}+E^{*}_{e^-})$", ax=axes[1, 0])
compare("cos_theta_l", np.linspace(-1, 1, 60),
        r"$\cos\theta_\ell$: $e^+$ (in $e^+e^-$ frame) vs dilepton flight dir. (in $\pi^0$ frame)", ax=axes[1, 1])
compare("m_ee", mee_bins, r"$m_{ee}$ [MeV]", logy=True, logx=True, ax=axes[1, 2])
fig.suptitle(r"$\pi^0 \to e^+e^-\gamma$: Geant4 vs EvtGen ($\pi^0$ rest frame)")
fig.tight_layout()
savefig("geant4_vs_evtgen_overview")
plt.show()

## $m_{ee}$ spectrum — the key discriminator (enlarged)

This is the **same** $e^+e^-$ invariant-mass distribution as the bottom-right panel above, shown enlarged on a log-log scale with the $2m_e$ threshold marked, to zoom in on the low-mass region.

The $m_{ee}$ spectrum is the most direct probe of the decay matrix element. The Kroll-Wada distribution rises steeply as $m_{ee}\to 2m_e$ (the $1/m_{ee}^2$ virtual-photon propagator), modulated by phase space and the transition form factor. A simplified Geant4 treatment can differ in shape here.

In [ ]:
bins = np.logspace(np.log10(2 * M_E), np.log10(M_PI0), 60)
fig, ax = plt.subplots(figsize=(7, 5))
ax.hist(g4_obs["m_ee"], bins=bins, density=True, histtype="step", lw=1.8,
        color="tab:blue", label=f"Geant4 (N={len(g4_obs['m_ee']):,})")
ax.hist(ev_obs["m_ee"], bins=bins, density=True, histtype="step", lw=1.8,
        color="tab:red", label=f"EvtGen (N={len(ev_obs['m_ee']):,})")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel(r"$m_{ee}$ [MeV]")
ax.set_ylabel("normalized")
ax.axvline(2 * M_E, ls=":", color="gray", label=r"$2m_e$ threshold")
ax.legend()
fig.tight_layout()
savefig("geant4_vs_evtgen_mee")
plt.show()